# Notebook 3 — PCA + Ridge (spot vs. slide vs. patient split)

**What this notebook does**
1. Loads the embeddings (Notebook 2) and the log1p expression (Notebook 1) for every spot, lined up by barcode.
2. Runs the **same baseline** under three ways of splitting train and test:
   - **spot split:** random spots go to test (neighbouring spots land on both sides → most leakage)
   - **slide split:** whole sections go to test (another section of the same patient can still be in training → some leakage)
   - **patient split:** whole patients go to test (all their sections) → the honest score
3. Inside every fold, using **training spots only**:
   1. choose the top 50 most variable genes
   2. fit StandardScaler + PCA (256 PCs)
   3. choose ridge's λ for each gene with an inner split that groups the same way as the outer split
   4. fit ridge, predict the test spots
4. Scores every gene on the test spots (PCC, RMSE, NRMSE), next to the mean predictor, and builds the **leakage table**: the same model's score under each split.

**Why folds:** with only a few patients, one train/test split depends on which patients happen to land in test. K-fold cross-validation lets every patient be tested once and reports the average.

**Where it runs:** Google Colab, **CPU** runtime is enough (no GPU needed).

**Before you run it**
- Notebooks 1 and 2 have finished, so `HEST_breast_ST/` has `spot_table.csv`, `genes.txt`, `expression/` and `embeddings/`.
- The shared folder `BDD_BJMJ` is visible in your My Drive.

**What it saves** (in `HEST_breast_ST/results/`)

| File | Contents |
|---|---|
| `fold_scores.csv` | One row per split × fold: train/test sizes, test sections and patients, mean PCC / RMSE / NRMSE for ridge and the mean predictor |
| `gene_scores.csv` | One row per split × fold × gene: PCC, RMSE, NRMSE and the λ chosen for that gene |
| `leakage_table.csv` | One row per split: average over folds (the main result) |
| `fold_assignments.csv` | Which fold each spot was in the test set for, per split |
| `settings.json` | The settings used for this run |

**To scale up:** after Notebooks 1 and 2 are re-run with all 23 patients, change `N_FOLDS = 4` in Settings. Nothing below it needs editing.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import scipy.sparse
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

**Connect Google Drive:** pick the Google account that can see `BDD_BJMJ`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Settings

- `SPLITS`: which splits to run. All three give the leakage table
- `N_FOLDS`: number of outer folds. It can't be more than the number of patients: **3** for the 3-patient test run, **4** for all 23 patients
- `N_INNER_FOLDS`: folds inside the training set for choosing λ (fewer if there aren't enough patients or sections)
- `N_GENES`: genes to predict, chosen by variance on training spots in each fold
- `N_PCS`: PCA components (256, as in HEST)
- `ALPHAS`: the λ values ridge tries, 0.01 to 100,000
- `MIN_TEST_SD`: genes whose test SD is below this are skipped for NRMSE (dividing by ~0 would blow up)

In [ ]:
TEAM_FOLDER = "/content/drive/MyDrive/BDD_BJMJ"
DATA_DIR = os.path.join(TEAM_FOLDER, "HEST_breast_ST")

SPLITS = ["spot", "slide", "patient"]
N_FOLDS = 3
N_INNER_FOLDS = 3

N_GENES = 50
N_PCS = 256
ALPHAS = np.logspace(-2, 5, 30)
MIN_TEST_SD = 0.05
SEED = 0

In [ ]:
if not os.path.isdir(TEAM_FOLDER):
    raise FileNotFoundError(
        f"Can't find {TEAM_FOLDER}. Open Google Drive → Shared with me, right-click BDD_BJMJ → "
        "Organize → Add shortcut → My Drive, then run the Drive cell and this cell again."
    )

EXPRESSION_DIR = os.path.join(DATA_DIR, "expression")
EMBEDDING_DIR = os.path.join(DATA_DIR, "embeddings")
RESULTS_DIR = os.path.join(DATA_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Reading from:", DATA_DIR)
print("Saving to:", RESULTS_DIR)

## Step 1. Load embeddings and expression

For each section we load its embeddings (Notebook 2) and its log1p expression (Notebook 1), then stack all sections. Both files have their rows in the same spot order, and we check the barcodes to be sure.

- `X`: embeddings, one row per spot, 4,608 columns (model input)
- `Y`: log1p expression, one row per spot, one column per shared gene (target). Kept sparse to save memory
- `spots`: the spot table rows in the same order (section, patient, barcode)

In [ ]:
spot_table = pd.read_csv(os.path.join(DATA_DIR, "spot_table.csv"))
with open(os.path.join(DATA_DIR, "genes.txt")) as f:
    all_genes = np.array(f.read().split())

X_parts = []
Y_parts = []
spot_parts = []

for sample_id in spot_table["sample_id"].unique():
    section_spots = spot_table[spot_table["sample_id"] == sample_id].sort_values("row_in_sample")

    saved = np.load(os.path.join(EMBEDDING_DIR, f"{sample_id}.npz"), allow_pickle=True)
    assert np.array_equal(saved["barcode"], section_spots["barcode"].to_numpy()), \
        f"{sample_id}: embedding rows don't match the spot table"

    expression = scipy.sparse.load_npz(os.path.join(EXPRESSION_DIR, f"{sample_id}.npz"))
    assert expression.shape == (len(section_spots), len(all_genes)), \
        f"{sample_id}: expression shape doesn't match the spot table and genes.txt"

    X_parts.append(saved["embeddings"])
    Y_parts.append(expression)
    spot_parts.append(section_spots)

X = np.concatenate(X_parts, axis=0)
Y = scipy.sparse.vstack(Y_parts).tocsc()
spots = pd.concat(spot_parts, ignore_index=True)

print("X (embeddings):", X.shape)
print("Y (log1p expression):", Y.shape)
print("Sections:", spots["sample_id"].nunique(), " Patients:", spots["patient"].nunique())

## Step 2. Make the folds for each split

Each spot gets a fold number. In fold *k*, the spots with fold *k* are the test set and everything else is training.

- **spot:** spots are shuffled and dealt into folds at random
- **slide:** all spots of a section get the same fold (`GroupKFold` grouped by `sample_id`)
- **patient:** all spots of a patient get the same fold (`GroupKFold` grouped by `patient`)

The check at the end makes sure no section (slide split) or patient (patient split) is ever in both train and test.

In [ ]:
def group_column(split):
    if split == "slide":
        return "sample_id"
    if split == "patient":
        return "patient"
    return None


def make_folds(split, n_folds):
    fold_of_spot = np.zeros(len(spots), dtype=int)
    column = group_column(split)

    if column is None:
        splitter = KFold(n_splits=n_folds, shuffle=True, random_state=SEED)
        fold_splits = splitter.split(X)
    else:
        n_groups = spots[column].nunique()
        if n_folds > n_groups:
            raise ValueError(f"N_FOLDS = {n_folds} but there are only {n_groups} groups for the {split} split.")
        splitter = GroupKFold(n_splits=n_folds)
        fold_splits = splitter.split(X, groups=spots[column])

    for fold, (train_index, test_index) in enumerate(fold_splits):
        fold_of_spot[test_index] = fold
    return fold_of_spot


fold_assignments = spots[["sample_id", "patient", "barcode"]].copy()
for split in SPLITS:
    fold_assignments[split] = make_folds(split, N_FOLDS)

for split in SPLITS:
    column = group_column(split)
    print(f"--- {split} split ---")
    for fold in range(N_FOLDS):
        is_test = fold_assignments[split] == fold
        test_patients = sorted(spots.loc[is_test, "patient"].unique())
        test_sections = sorted(spots.loc[is_test, "sample_id"].unique())
        print(f"fold {fold}: {is_test.sum()} test spots | test patients: {test_patients} | test sections: {len(test_sections)}")
        if column is not None:
            train_groups = set(spots.loc[~is_test, column])
            test_groups = set(spots.loc[is_test, column])
            assert len(train_groups & test_groups) == 0, f"{split} fold {fold}: a {column} is in both train and test"

fold_assignments.to_csv(os.path.join(RESULTS_DIR, "fold_assignments.csv"), index=False)

## Step 3. The pieces of one fold

**Choose genes (training spots only).** Variance of each gene's log1p expression across training spots; keep the top `N_GENES`. Using all spots here would let the test spots influence which genes we predict, a small leak.

**Choose λ per gene (inner split).** Inside the training set we split again, grouped the same way as the outer split: by patient for the patient split, by section for the slide split, random for the spot split. For each λ we train on the inner-train part and measure the error on the inner-validation part, and each gene keeps the λ with the lowest average error. The test spots are never used to choose λ.

The inner split works on the PCA scores from the outer training set. PCA sees no gene values, so this is a common simplification; the test set stays completely untouched.

**Score.** For each gene on the test spots:
- **PCC:** does the prediction go up where the gene really goes up? (undefined if either side is constant)
- **RMSE:** average size of the error, in log1p units
- **NRMSE** = RMSE ÷ the gene's test SD: 1 ≈ no better than predicting the average

In [ ]:
def choose_genes(train_index):
    Y_train = Y[train_index]
    mean = np.asarray(Y_train.mean(axis=0)).ravel()
    mean_of_squares = np.asarray(Y_train.multiply(Y_train).mean(axis=0)).ravel()
    variance = mean_of_squares - mean ** 2
    return np.argsort(variance)[::-1][:N_GENES]


def choose_alpha_per_gene(P_train, y_train, inner_groups):
    if inner_groups is None:
        n_inner = N_INNER_FOLDS
        inner_splits = KFold(n_splits=n_inner, shuffle=True, random_state=SEED).split(P_train)
    else:
        n_inner = min(N_INNER_FOLDS, len(np.unique(inner_groups)))
        inner_splits = GroupKFold(n_splits=n_inner).split(P_train, groups=inner_groups)

    errors = np.zeros((len(ALPHAS), y_train.shape[1]))
    for inner_train, inner_val in inner_splits:
        for a, alpha in enumerate(ALPHAS):
            model = Ridge(alpha=alpha)
            model.fit(P_train[inner_train], y_train[inner_train])
            prediction = model.predict(P_train[inner_val])
            errors[a] += np.mean((prediction - y_train[inner_val]) ** 2, axis=0)

    best = np.argmin(errors, axis=0)
    return ALPHAS[best]


def fit_and_predict(P_train, y_train, P_test, alpha_per_gene):
    y_pred = np.zeros((P_test.shape[0], y_train.shape[1]))
    for alpha in np.unique(alpha_per_gene):
        genes_with_this_alpha = np.where(alpha_per_gene == alpha)[0]
        model = Ridge(alpha=alpha)
        model.fit(P_train, y_train[:, genes_with_this_alpha])
        prediction = model.predict(P_test).reshape(len(P_test), -1)
        y_pred[:, genes_with_this_alpha] = prediction
    return y_pred


def pcc_one_gene(true_values, predicted_values):
    if np.std(true_values) < 1e-6 or np.std(predicted_values) < 1e-6:
        return np.nan
    return np.corrcoef(true_values, predicted_values)[0, 1]


def score_genes(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    sd = np.std(y_true, axis=0)
    nrmse = np.where(sd >= MIN_TEST_SD, rmse / np.maximum(sd, 1e-12), np.nan)
    pcc = np.array([pcc_one_gene(y_true[:, g], y_pred[:, g]) for g in range(y_true.shape[1])])
    return rmse, nrmse, pcc

**One fold, start to finish.** Every step that learns from data (genes, scaler, PCA, λ, ridge) is fit on `train_index` only, then applied to `test_index`.

In [ ]:
def run_one_fold(split, fold):
    is_test = fold_assignments[split].to_numpy() == fold
    train_index = np.where(~is_test)[0]
    test_index = np.where(is_test)[0]

    gene_index = choose_genes(train_index)
    y_train = Y[train_index][:, gene_index].toarray()
    y_test = Y[test_index][:, gene_index].toarray()

    scaler = StandardScaler()
    Z_train = scaler.fit_transform(X[train_index])
    Z_test = scaler.transform(X[test_index])

    n_components = min(N_PCS, Z_train.shape[0], Z_train.shape[1])
    pca = PCA(n_components=n_components, random_state=SEED)
    P_train = pca.fit_transform(Z_train)
    P_test = pca.transform(Z_test)

    column = group_column(split)
    inner_groups = None if column is None else spots[column].to_numpy()[train_index]
    alpha_per_gene = choose_alpha_per_gene(P_train, y_train, inner_groups)

    y_pred_ridge = fit_and_predict(P_train, y_train, P_test, alpha_per_gene)
    y_pred_mean = np.tile(y_train.mean(axis=0), (len(test_index), 1))

    rmse, nrmse, pcc = score_genes(y_test, y_pred_ridge)
    rmse_mean, nrmse_mean, _ = score_genes(y_test, y_pred_mean)

    gene_rows = pd.DataFrame({
        "split": split,
        "fold": fold,
        "gene": all_genes[gene_index],
        "PCC": pcc,
        "RMSE": rmse,
        "NRMSE": nrmse,
        "alpha": alpha_per_gene,
    })

    fold_row = {
        "split": split,
        "fold": fold,
        "n_train": len(train_index),
        "n_test": len(test_index),
        "n_pcs": n_components,
        "test_patients": " ".join(sorted(spots["patient"].iloc[test_index].unique())),
        "test_sections": " ".join(sorted(spots["sample_id"].iloc[test_index].unique())),
        "ridge_PCC": np.nanmean(pcc),
        "ridge_PCC_median": np.nanmedian(pcc),
        "ridge_RMSE": np.mean(rmse),
        "ridge_NRMSE": np.nanmean(nrmse),
        "mean_predictor_RMSE": np.mean(rmse_mean),
        "mean_predictor_NRMSE": np.nanmean(nrmse_mean),
        "median_alpha": np.median(alpha_per_gene),
    }
    return fold_row, gene_rows

## Step 4. Run every split and fold

In [ ]:
fold_rows = []
gene_tables = []

for split in SPLITS:
    for fold in range(N_FOLDS):
        fold_row, gene_rows = run_one_fold(split, fold)
        fold_rows.append(fold_row)
        gene_tables.append(gene_rows)
        print(f"{split:8s} fold {fold}: train {fold_row['n_train']:6d}  test {fold_row['n_test']:6d}  "
              f"PCC {fold_row['ridge_PCC']:.3f}  NRMSE {fold_row['ridge_NRMSE']:.3f}", flush=True)

fold_scores = pd.DataFrame(fold_rows)
gene_scores = pd.concat(gene_tables, ignore_index=True)

## Step 5. The leakage table

The same model, scored three ways. Each number is the average over folds (± SD across folds).

What to expect: **spot > slide > patient**. The drop from spot to patient is how much of the spot-split score came from spatial leakage. The patient-split number is the honest one.

In [ ]:
leakage_rows = []
for split in SPLITS:
    rows = fold_scores[fold_scores["split"] == split]
    leakage_rows.append({
        "split": split,
        "folds": len(rows),
        "PCC_mean": rows["ridge_PCC"].mean(),
        "PCC_sd": rows["ridge_PCC"].std(),
        "NRMSE_mean": rows["ridge_NRMSE"].mean(),
        "NRMSE_sd": rows["ridge_NRMSE"].std(),
        "RMSE_mean": rows["ridge_RMSE"].mean(),
        "mean_predictor_NRMSE": rows["mean_predictor_NRMSE"].mean(),
    })

leakage_table = pd.DataFrame(leakage_rows)
print(leakage_table.round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x_positions = np.arange(len(SPLITS))

for ax, metric, title in [(axes[0], "ridge_PCC", "Mean test PCC (higher is better)"),
                          (axes[1], "ridge_NRMSE", "Mean test NRMSE (lower is better)")]:
    means = [fold_scores.loc[fold_scores["split"] == s, metric].mean() for s in SPLITS]
    ax.bar(x_positions, means, color="lightgray", edgecolor="black")
    for i, split in enumerate(SPLITS):
        fold_values = fold_scores.loc[fold_scores["split"] == split, metric]
        ax.scatter(np.full(len(fold_values), i), fold_values, color="black", zorder=3, s=20)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(SPLITS)
    ax.set_title(title)

axes[1].axhline(1.0, color="red", linestyle="--", label="mean predictor ≈ 1")
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "leakage_plot.png"), dpi=150)
plt.show()

**Best and worst genes under the patient split** (PCC averaged over folds, for genes chosen in at least one fold).

In [ ]:
if "patient" in SPLITS:
    patient_genes = gene_scores[gene_scores["split"] == "patient"]
    gene_summary = patient_genes.groupby("gene").agg(PCC=("PCC", "mean"), folds=("fold", "count"))
    gene_summary = gene_summary.sort_values("PCC", ascending=False)
    print("Top 10:")
    print(gene_summary.head(10).round(3).to_string())
    print()
    print("Bottom 10:")
    print(gene_summary.tail(10).round(3).to_string())

## Step 6. Save everything

In [ ]:
fold_scores.to_csv(os.path.join(RESULTS_DIR, "fold_scores.csv"), index=False)
gene_scores.to_csv(os.path.join(RESULTS_DIR, "gene_scores.csv"), index=False)
leakage_table.to_csv(os.path.join(RESULTS_DIR, "leakage_table.csv"), index=False)

settings = {
    "splits": SPLITS,
    "n_folds": N_FOLDS,
    "n_inner_folds": N_INNER_FOLDS,
    "n_genes": N_GENES,
    "n_pcs": N_PCS,
    "alphas": [float(a) for a in ALPHAS],
    "min_test_sd": MIN_TEST_SD,
    "seed": SEED,
    "n_spots": int(len(spots)),
    "n_sections": int(spots["sample_id"].nunique()),
    "n_patients": int(spots["patient"].nunique()),
    "patients": sorted(spots["patient"].unique().tolist()),
}
with open(os.path.join(RESULTS_DIR, "settings.json"), "w") as f:
    json.dump(settings, f, indent=2)

print("Saved to", RESULTS_DIR)
print(sorted(os.listdir(RESULTS_DIR)))

## How to read the test-run numbers

With 3 patients and 3 folds, the patient split trains on only **2 patients** each time, so its score will be low and noisy. That's expected. The test run is for checking that the pipeline works end to end; the real leakage result comes from all 23 patients with `N_FOLDS = 4`.